Example 8.4

In [ ]:
def create_or_update_table(table_name, table_schema, recreate):
    """ Function for creating table or updating it with new columns """

    if recreate:
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")

    columns_definitions = [
        f"`{column_name}` {column_type}"
        for column_name, column_type in table_schema.items()
    ]

    columns_definitions_string = ', '.join(columns_definitions)

    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {table_name} "
        f"({columns_definitions_string})"
    )

    table_columns = {
        column.name for column in spark.catalog.listColumns(table_name)
    }

    missings = [
    (column_name, column_type)
        for column_name, column_type in table_schema.items()
        if column_name not in table_columns
    ]

    if missings:
        new_columns = ", ".join(
            f"{column_name} {column_type}" for column_name, column_type in missings
        )

    spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS ({new_columns})")

Example 8-14

In [ ]:
from pyspark.sql import functions as F

customers_df = spark.table("silver.dbo.customers")
orders_df = spark.table("silver.dbo.orders")

salt_buckets = 16

orders_df = orders_df.withColumn(
    "salt",
    #F.pmod(F.xxhash64("order_id", "customer_id"), F.lit(salt_buckets))
    F.pmod(F.col("order_id"), F.lit(salt_buckets))
)

customers_df = customers_df.withColumn(
    "salt",
    F.explode(F.sequence(F.lit(0), F.lit(salt_buckets - 1)))
)

sales_df = customers_df.join(
    orders_df,
    (customers_df["id"] == orders_df["customer_id"]) &
    (customers_df["salt"] == orders_df["salt"]),
    "inner"
)

Example 8-27

In [ ]:
import pyspark.sql.functions as F

DATA_PATH = 'Files/images'

df_images = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.png") \
    .option("recursiveFileLookup", "false") \
    .load(DATA_PATH) \
    .withColumn("filename", F.expr("split(path, '/')[size(split(path, '/')) - 1]")) \
    .withColumn("id", F.expr("split(filename, '_')[1]").substr(1, 4).cast("int")) \
    .select("id", "content", "length", "modificationTime", "year", "month", "day" )

df_images.show()

Example 8-36

In [ ]:
import requests
import pyarrow as pa
from deltalake import write_deltalake, DeltaTable

API_URL = "https://apidatasource.azurewebsites.net/api/Customers"
resp = requests.get(API_URL)
data = resp.json()

schema = pa.schema([
    ("id", pa.int64()),
    ("name", pa.string()),
    ("email", pa.string()),
])

table = pa.Table.from_pylist(data, schema=schema)

TABLE_PATH = "abfss://DEGTMF@onelake.dfs.fabric.microsoft.com/silver.Lakehouse/Tables/customers_5"
write_deltalake(table_or_uri=TABLE_PATH, data=table, mode="overwrite")

Example 8-38

In [ ]:
import struct
import pyodbc
import pandas as pd

SERVER = ""
DATABASE = "silver"
CONNECTION_STRING = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={SERVER};DATABASE={DATABASE};Encrypt=yes"

def _pack_token(s: str) -> bytes:
    encoded = s.encode("utf-8")
    body = b"".join(bytes((b, 0)) for b in encoded)
    return struct.pack("=i", len(body)) + body
    access_token = notebookutils.credentials.getToken("pbi")
    attrs_before={
        1256: _pack_token(access_token) # SQL_COPT_SS_ACCESS_TOKEN
    }

with pyodbc.connect(CONNECTION_STRING, attrs_before=attrs_before) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM [silver].[dbo].[customers]")
    rows = cursor.fetchall()
    df = pd.DataFrame.from_records(rows, columns=[column[0] for column in cursor.description])

Example 8-39

In [ ]:
import com.microsoft.sqlserver.jdbc.spark
import pyspark.sql.functions as F


URL = "jdbc:sqlserver://mydatabase.database.windows.net;database=customers;"

df = spark.read.option("URL", url).mssql("dbo.customers")
df = df.filter(F.col("country") == F.lit("Germany"))
df.write.mode("overwrite").option("URL", url).mssql("dbo.german_customers")